# 00 - First contact with YOLO

**Goal:** run a pretrained YOLO detector, learn what its output looks like, then study the drone dataset (VisDrone) used for the whole project and see why aerial detection is hard.

**Before you run:** in the notebook settings, pick a GPU under *Accelerator* and switch *Internet* on.

| Part | What happens |
|---|---|
| A | Check the GPU and install the library |
| B | Run a pretrained model on one image and read its output |
| C | Download VisDrone and measure how small the objects are |
| D | See what a model trained on everyday photos does on a drone image |

## Part A - Setup

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q ultralytics

In [ ]:
import ultralytics
from ultralytics import YOLO

ultralytics.checks()  # prints Python, PyTorch, GPU, RAM and disk information

## Part B - A pretrained model on one image

`yolo26n.pt` is a YOLO26 **nano** model (the smallest and fastest size) trained on COCO, a dataset of everyday photos with 80 object classes. The weights download automatically the first time.

In [ ]:
model = YOLO("yolo26n.pt")

print("Task:", model.task)
print("Number of classes:", len(model.names))
print("First 10 classes:", list(model.names.values())[:10])

In [ ]:
# Run detection on a sample image from a URL. The result is a list with one entry per image.
results = model("https://ultralytics.com/images/bus.jpg")
r = results[0]

print("Image size (height, width):", r.orig_shape)
print("Detections:", len(r.boxes))

Each detection has three things: a **class**, a **confidence** score and a **box**. Boxes come as `xyxy`, meaning top-left corner `(x1, y1)` and bottom-right corner `(x2, y2)` in pixels.

In [ ]:
for box in r.boxes:
    class_name = model.names[int(box.cls.item())]
    confidence = box.conf.item()
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    print(f"{class_name:>8} | confidence {confidence:.2f} | box (x1, y1, x2, y2) = ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 8))
plt.imshow(r.plot()[..., ::-1])  # plot() draws the boxes and returns BGR; [..., ::-1] flips it to RGB
plt.axis("off")
plt.show()

**Experiment:** the model throws away detections whose confidence is below a threshold. Watch the detection count change as the threshold moves.

In [ ]:
img = r.orig_img  # the image as a NumPy array
for threshold in (0.05, 0.25, 0.5, 0.8):
    n = len(model(img, conf=threshold, verbose=False)[0].boxes)
    print(f"confidence threshold {threshold}: {n} detections")

## Part C - The drone dataset

VisDrone is a benchmark of images taken from drones. The next cell downloads it (a few GB, so it can take several minutes) and converts its labels to YOLO format. If this cell fails, copy the full error message and send it to me.

In [ ]:
from pathlib import Path
from ultralytics import settings
from ultralytics.data.utils import check_det_dataset

settings.update({"datasets_dir": str(Path.cwd() / "datasets")})  # store data in the writable working folder

data = check_det_dataset("VisDrone.yaml")
print("Dataset folder:", data["path"])
print("Classes:", data["names"])

YOLO label files hold one line per object: `class  x_center  y_center  width  height`, with all four numbers **normalized** to 0-1 by the image size. The next cell converts them back to pixels and measures how big the objects really are.

In [ ]:
import numpy as np
from PIL import Image

val_images = Path(data["val"])
val_labels = Path(data["path"]) / "labels" / "val"  # layout is images/val and labels/val

image_paths, objects_per_image, class_ids, widths, heights = [], [], [], [], []
for img_path in sorted(val_images.glob("*.jpg")):
    label_file = val_labels / f"{img_path.stem}.txt"
    if not label_file.exists():
        continue
    img_w, img_h = Image.open(img_path).size
    rows = [line.split() for line in label_file.read_text().splitlines() if line.strip()]
    image_paths.append(img_path)
    objects_per_image.append(len(rows))
    for cls, cx, cy, bw, bh in rows:
        class_ids.append(int(cls))
        widths.append(float(bw) * img_w)
        heights.append(float(bh) * img_h)

sizes = np.sqrt(np.array(widths) * np.array(heights))  # one "how big is it" number per object, in pixels

print("Images:", len(image_paths))
print("Objects:", len(class_ids))
print("Objects per image: mean", round(float(np.mean(objects_per_image)), 1), "| max", max(objects_per_image))
print("Median object size:", round(float(np.median(sizes)), 1), "pixels")
print(f"Small  (under 32 px): {np.mean(sizes < 32):.0%}")
print(f"Medium (32 to 96 px): {np.mean((sizes >= 32) & (sizes < 96)):.0%}")
print(f"Large  (over 96 px):  {np.mean(sizes >= 96):.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(sizes, bins=60, range=(0, 200))
axes[0].set(title="Object size distribution", xlabel="size in pixels (square root of box area)", ylabel="number of objects")

counts = np.bincount(class_ids, minlength=len(data["names"]))
axes[1].bar([data["names"][i] for i in range(len(counts))], counts)
axes[1].set(title="Objects per class (validation set)")
axes[1].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

## Part D - Everyday-photo model meets drone footage

We take the most crowded validation image and run the COCO-pretrained model at two input sizes. The `imgsz` setting is the size the image is resized to before the network sees it. Compare the detections with the number of real objects.

In [ ]:
busiest = image_paths[int(np.argmax(objects_per_image))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, size in zip(axes, (640, 1280)):
    res = model(str(busiest), imgsz=size, verbose=False)[0]
    ax.imshow(res.plot()[..., ::-1])
    ax.set_title(f"imgsz={size}: {len(res.boxes)} detections")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Ground-truth objects in this image:", max(objects_per_image))

## Your Day 1 notes

Write short answers in a markdown cell below, in your own words:
1. What fraction of VisDrone objects are small, and why does that make detection hard?
2. What did the pretrained model miss on the drone image? What changed at the larger `imgsz`?
3. What does the confidence threshold trade off?

Next: IoU and mAP (how we score a detector), then a first baseline training run.